# Ask the OKF bundle: questions → answers from your local LLM

This notebook answers questions from **`OKF_Bundle/`** using your local model in Ollama (`qwen3:4b-instruct-2507-q4_K_M`).

**Flow:** load bundle → BM25 index → retrieve (topic pages → sections → follow links) → build context → ask the LLM → show answer + sources + timing.

### What is in the bundle?

OKF (Open Knowledge Format) is just **a folder of markdown files with YAML frontmatter**. One file = one *concept*. Markdown links between files make the knowledge graph. There is no database or embeddings, and you don't need an SDK.

This bundle has **two layers**:

| Layer | Folder | `type` | What it is |
|---|---|---|---|
| Sections | `papers/<paper>/…` | `Paper Section` | The paper text, split by section heading (long sections → `part-1`, `part-2`…). Each file knows its paper, section path and **page numbers**. |
| Wiki | `topics/…` and `papers/<paper>/overview.md` | `Model`, `Method`, `Benchmark`, `Metric`, `Concept`, `Paper` | Curated pages that summarise what *all five papers* say about one thing, with links to the exact sections. |

### How a question is answered

```
question
  │
  ├─► 1. BM25 over the wiki layer      → best topic page(s)        "what is this about?"
  ├─► 2. BM25 over the section layer   → best section(s)           "where is it said?"
  ├─► 3. follow links from the topics  → sections they cite        "the evidence the wiki points to"
  │
  ├─► 4. pack those concepts into a context (character budget)
  └─► 5. qwen3-4b answers ONLY from the context and cites concept IDs
```

**How this differs from your `01_Chunking.ipynb` RAG:** there, the unit is a ~1,200-character chunk found by dense + sparse vectors in Qdrant, then reranked with ColBERT. Here, the unit is a whole **concept** (a section, or a curated page). Search is plain BM25 in pure Python. The "graph" step just follows the markdown links that were written when the bundle was built.

## 1. Setup

Everything you might want to tune is in this one cell.
- `NUM_CTX` is the model's context window in tokens. Ollama's default is small, so we set it. 8192 is safe on a 6 GB GPU.
- `CONTEXT_CHARS` is how much bundle text we send. Roughly 4 characters ≈ 1 token, so 18,000 chars ≈ 4.5k tokens. That leaves room for the question and the answer inside 8192.

In [1]:
import json
import math
import re
import time
from collections import Counter, defaultdict
from pathlib import Path
import pandas as pd

import requests
import yaml

# works from the project folder or from inside OKF_Bundle/
BUNDLE_DIR = Path.cwd() if (Path.cwd() / "index.md").exists() else Path.cwd() / "OKF_Bundle"

OLLAMA_URL = "http://localhost:11434"
MODEL_NAME = "qwen3:4b-instruct-2507-q4_K_M"
NUM_CTX = 8192            # tokens the model can see (prompt + answer)
MAX_ANSWER_TOKENS = 700
CONTEXT_CHARS = 18000     # budget for bundle text in the prompt
MAX_CHARS_PER_CONCEPT = 4500

# retrieval knobs (step 5)
N_TOPICS = 2      # wiki pages
N_SECTIONS = 2    # best-matching paper sections
N_LINKED = 2      # extra sections reached by following links from the wiki pages

print("bundle:", BUNDLE_DIR)
assert (BUNDLE_DIR / "index.md").exists(), "index.md not found - run this notebook from the project folder (next to OKF_Bundle/)"

bundle: c:\Users\Swapn\OneDrive\Documents\GitHub\OKF\OKF_Bundle


In [2]:
# check: Ollama is running and the model is installed
tags = requests.get(f"{OLLAMA_URL}/api/tags", timeout=10).json()
installed = [m["name"] for m in tags.get("models", [])]
print("installed:", installed)
assert MODEL_NAME in installed, f"{MODEL_NAME} not installed. Run: ollama pull {MODEL_NAME}"

installed: ['qwen3:4b-instruct-2507-q4_K_M', 'qwen2.5-0.5b-instruct-fc:latest', 'qwen2.5:0.5b-instruct', 'llama3.2:3b', 'qwen3:4b-instruct']


## 2. What one concept file looks like

Before writing any code, read a file. The top block between `---` lines is the **frontmatter** (YAML):
- `type` is the only field OKF *requires*.
- `title`, `description` and `tags` are recommended.
- `sources` (provenance), `generated` (who wrote it, when) and `status` come from the v0.2 trust fields. `aliases` is an extra field this bundle adds, and OKF allows extra fields.

Under the frontmatter is ordinary markdown. The links like `[...](/papers/helm/4-4-calibration-and-uncertainty.md)` **are** the knowledge graph.

In [3]:
print((BUNDLE_DIR / "topics" / "calibration.md").read_text(encoding="utf-8")[:2500])

---
type: Metric
title: Calibration
description: Whether a model's confidence matches how often it is actually right; measured in HELM with expected calibration error (ECE) and selective classification.
tags: [metric, helm, uncertainty, evaluation]
aliases: [calibration and uncertainty, ECE, expected calibration error, confidence calibration, selective classification]
status: draft
generated: {by: "claude/claude-opus-5-5", at: "2026-10-11T15:23:14Z"}
sources:
  - id: s1
    resource: "/papers/helm/4-4-calibration-and-uncertainty.md"
    title: "Holistic Evaluation of Language Models: 4.4 Calibration and uncertainty (pp. 27–28)"
  - id: s2
    resource: "/papers/helm/c-2-calibration-and-uncertainty-part-1.md"
    title: "Holistic Evaluation of Language Models: C.2 Calibration and uncertainty (part 1 of 2) (pp. 131–133)"
  - id: s3
    resource: "/papers/helm/1-2-empirical-findings-part-1.md"
    title: "Holistic Evaluation of Language Models: 1.2 Empirical findings (part 1 of 4) (pp. 6–

And a section concept. It has the paper text with `[p. N]` markers wherever the page changes, and a `## Links` block at the bottom: up/down/previous/next in the paper, plus **backlinks** to every topic page that cites this section.

In [4]:
text = (BUNDLE_DIR / "papers" / "helm" / "4-4-calibration-and-uncertainty.md").read_text(encoding="utf-8")
print(text[:1200], "\n...\n", text[text.index("## Links"):])

---
type: Paper Section
title: "4.4 Calibration and uncertainty"
description: "HELM, 4 General metrics › 4.4 Calibration and uncertainty, pp. 27–28."
tags: [helm]
paper: "Holistic Evaluation of Language Models"
section_path: ["4 General metrics", "4.4 Calibration and uncertainty"]
pages: [27, 28]
sources:
  - id: pdf
    resource: "../../../data/raw/Holistic_Eval.pdf#page=27"
    title: "Holistic Evaluation of Language Models, pp. 27–28"
generated: {by: "process:okf-section-split", at: "2026-10-11T15:23:14Z"}
---

# 4.4 Calibration and uncertainty

*Holistic Evaluation of Language Models › 4 General metrics › 4.4 Calibration and uncertainty · pp. 27–28*

[p. 27]

When machine learning models are integrated into broader systems, it is critical for these models to be simultaneously accurate (i.e. frequently correct) and able to express their uncertainty (so that their errors can be appropriately anticipated and accommodated). Calibration and appropriate expression of model uncertainty is

## 3. Load the bundle

Loading OKF takes very little code: walk the folder, split each file into frontmatter and body, and parse the frontmatter with `yaml.safe_load`.

`index.md` and `log.md` are **reserved filenames** in the spec. They are a table of contents and a change log, not concepts, so we skip them. The **concept ID** is the file path without `.md`, e.g. `topics/calibration`.

In [5]:
RESERVED = {"index.md", "log.md"}


def load_bundle(bundle_dir):
    concepts = {}
    for path in sorted(bundle_dir.rglob("*.md")):
        if path.name in RESERVED:
            continue
        text = path.read_text(encoding="utf-8")
        _, frontmatter, body = text.split("---\n", 2)          # '---' / yaml / '---' / markdown
        fm = yaml.safe_load(frontmatter)
        cid = path.relative_to(bundle_dir).with_suffix("").as_posix()
        concepts[cid] = {
            "id": cid,
            "type": fm["type"],
            "title": fm.get("title", cid),
            "description": fm.get("description", ""),
            "tags": fm.get("tags") or [],
            "aliases": fm.get("aliases") or [],
            "paper": fm.get("paper"),            # only on Paper Section concepts
            "pages": fm.get("pages"),            # [first, last]
            "body": body,
        }
    return concepts


start = time.perf_counter()
concepts = load_bundle(BUNDLE_DIR)
print(f"loaded {len(concepts)} concepts in {(time.perf_counter() - start) * 1000:.0f} ms")

loaded 857 concepts in 14408 ms


In [7]:
# check: what types are in the bundle?
for t, n in Counter(c["type"] for c in concepts.values()).most_common():
    print(f"{n:5}  {t}")

  798  Paper Section
   20  Concept
   15  Method
    8  Model
    6  Metric
    5  Paper
    5  Benchmark


## 4. The knowledge graph = markdown links

Every `[text](/some/concept.md)` in a body is a directed edge. We collect:
- `out_links[A]`: the concepts A points to. A topic page points to the sections it cites.
- `in_links[A]`: the concepts that point to A. For a section, these are its parent/sibling sections and the topic pages that cite it.

This is the step GraphRAG does with an LLM (entity and relation extraction). In OKF the edges were written once, when the bundle was built, so reading them is just a regex.

In [8]:
LINK_RE = re.compile(r"\[([^\]]*)\]\((/[^)#\s]+)\.md\)")     # [text](/path/to/concept.md)


def build_graph(concepts):
    out_links, in_links = defaultdict(set), defaultdict(set)
    for cid, c in concepts.items():
        for _, target in LINK_RE.findall(c["body"]):
            t = target.lstrip("/")
            if t in concepts and t != cid:
                out_links[cid].add(t)
                in_links[t].add(cid)
    return out_links, in_links


out_links, in_links = build_graph(concepts)
print("edges:", sum(len(v) for v in out_links.values()))

edges: 4408


In [9]:
# check: neighbours of one topic page
cid = "topics/calibration"
print(f"{cid} links OUT to {len(out_links[cid])} concepts:")
for t in sorted(out_links[cid]):
    print("   ->", t)
print(f"\nand is linked FROM {len(in_links[cid])} concepts, e.g.:", sorted(in_links[cid])[:6])

topics/calibration links OUT to 10 concepts:
   -> papers/gpt-4-technical-report/5-limitations
   -> papers/gpt-4-technical-report/6-risks-mitigations-part-1
   -> papers/helm/1-2-empirical-findings-part-1
   -> papers/helm/4-4-calibration-and-uncertainty
   -> papers/helm/c-2-calibration-and-uncertainty-part-1
   -> papers/sparks-of-agi/10-2-on-the-path-to-more-general-artificial-intelligenc-part-1
   -> topics/hallucination
   -> topics/helm-scenarios-and-metrics
   -> topics/overreliance
   -> topics/rlhf

and is linked FROM 14 concepts, e.g.: ['papers/gpt-4-technical-report/5-limitations', 'papers/gpt-4-technical-report/6-risks-mitigations-part-1', 'papers/gpt-4-technical-report/overview', 'papers/helm/1-2-empirical-findings-part-1', 'papers/helm/4-4-calibration-and-uncertainty', 'papers/helm/c-2-calibration-and-uncertainty-part-1']


## 5. Search: BM25

BM25 is classic keyword search, the same scoring as the `sparse` vectors in your Qdrant setup. For each query word it asks:
- **How rare is the word?** (IDF.) "calibration" is rare, so it counts a lot. "model" is everywhere, so it counts little.
- **How often is it in this concept?** (TF.) More is better, but with diminishing returns, controlled by `k1`.
- **How long is the concept?** Long documents are penalised a bit, controlled by `b`, so they don't win just by having more words.

**Field weighting** is where OKF frontmatter helps. Words in the `title` count 3×, and words in `aliases`, `tags` and `description` count 2×. So a page *titled* "TruthfulQA" beats a long section that only mentions it.

In [10]:
STOPWORDS = set('''a an the of and or to in on for with by from at as is are was were be been being it its this that these
those what which who whom how why when where do does did can could should would will may might about into than then
there their they them we our you your i me my he she his her not no yes vs'''.split())


def tokenize(text):
    # keeps 'gpt-4', 'text-davinci-002', '4.4' together as single tokens
    return [t for t in re.findall(r"[a-z0-9]+(?:[-.][a-z0-9]+)*", text.lower()) if t not in STOPWORDS]


class BM25:
    def __init__(self, concepts, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.tf, self.length = {}, {}
        for cid, c in concepts.items():
            body = LINK_RE.sub(r"\1", c["body"].split("\n## Links")[0])     # link text yes, URLs no
            fields = ((c["title"] + " ") * 3
                      + (" ".join(c["aliases"] + c["tags"]) + " " + c["description"] + " ") * 2
                      + body)
            tokens = tokenize(fields)
            self.tf[cid] = Counter(tokens)
            self.length[cid] = len(tokens)
        self.avg_len = sum(self.length.values()) / len(self.length)
        df = Counter(word for tf in self.tf.values() for word in tf)              # in how many concepts?
        n = len(self.tf)
        self.idf = {w: math.log(1 + (n - d + 0.5) / (d + 0.5)) for w, d in df.items()}

    def scores(self, query):
        q = tokenize(query)
        result = {}
        for cid, tf in self.tf.items():
            norm = 1 - self.b + self.b * self.length[cid] / self.avg_len
            s = sum(self.idf[w] * tf[w] * (self.k1 + 1) / (tf[w] + self.k1 * norm) for w in q if w in tf)
            if s > 0:
                result[cid] = s
        return result


start = time.perf_counter()
index = BM25(concepts)
print(f"indexed {len(index.tf)} concepts, vocabulary {len(index.idf)} words, in {(time.perf_counter() - start) * 1000:.0f} ms")

indexed 857 concepts, vocabulary 21387 words, in 371 ms


In [11]:
# check: plain BM25 top 8 for a question
q = "Which model was most accurate on TruthfulQA in HELM?"
s = index.scores(q)
for cid in sorted(s, key=s.get, reverse=True)[:8]:
    print(f"{s[cid]:6.2f}  {concepts[cid]['type']:14} {cid}")

 14.65  Paper Section  papers/helm/8-1-meta-analysis-part-4
 14.22  Paper Section  papers/helm/8-3-task-specific-results-for-core-scenarios-part-1
 14.12  Paper Section  papers/helm/8-4-targeted-evaluations-part-1
 13.74  Paper Section  papers/helm/1-2-empirical-findings-part-3
 12.85  Paper Section  papers/helm/8-1-meta-analysis-part-5
 11.44  Paper Section  papers/helm/1-2-empirical-findings-part-2
 11.39  Model          topics/instructgpt
 11.00  Paper Section  papers/helm/8-2-prompting-analysis-part-3


Look at the result above. Long HELM *results* sections can outrank the curated **TruthfulQA** and **InstructGPT** pages, because they repeat "model", "accurate" and "HELM" many times. That's why the next step searches the two layers separately.

## 6. Retrieve: wiki first, then evidence, then follow links

1. **Topics:** the best `N_TOPICS` wiki pages (types other than `Paper Section`). A second topic is kept only if it scores at least half as well as the first, so weak matches don't take up space.
2. **Sections:** the best `N_SECTIONS` paper sections, i.e. raw text with page numbers.
3. **Links:** sections that the chosen topic pages *link to*, ranked by their BM25 score for this question. This is the graph step. It brings in evidence that a wiki page vouches for, even when that section doesn't rank in the top `N_SECTIONS` on its own.

In [12]:
def is_wiki(cid):
    return concepts[cid]["type"] != "Paper Section"


def retrieve(question, n_topics=N_TOPICS, n_sections=N_SECTIONS, n_linked=N_LINKED):
    s = index.scores(question)
    ranked = sorted(s, key=s.get, reverse=True)
    topics = [c for c in ranked if is_wiki(c)]
    sections = [c for c in ranked if not is_wiki(c)]

    picked = []
    if topics:
        best = s[topics[0]]
        picked += [(c, "topic") for c in topics[:n_topics] if s[c] >= 0.5 * best]
    picked += [(c, "section") for c in sections[:n_sections]]

    already = {c for c, _ in picked}
    candidates = {t for c, via in picked if via == "topic" for t in out_links[c] if not is_wiki(t)} - already
    linked = sorted((c for c in candidates if c in s), key=s.get, reverse=True)[:n_linked]
    picked += [(c, "link") for c in linked]
    return [(c, via, round(s.get(c, 0), 2)) for c, via in picked]


for cid, via, score in retrieve("Which model was most accurate on TruthfulQA in HELM?"):
    print(f"{via:8} {score:6.2f}  {cid}")

topic     11.39  topics/instructgpt
topic     10.54  topics/truthfulqa
section   14.65  papers/helm/8-1-meta-analysis-part-4
section   14.22  papers/helm/8-3-task-specific-results-for-core-scenarios-part-1
link      13.74  papers/helm/1-2-empirical-findings-part-3
link      10.42  papers/helm/b-1-8-truthfulqa


## 7. Build the context

Each picked concept goes into the prompt with a header carrying its **ID**, which the model will cite, and, for sections, the **paper and pages**.

Two clean-ups save tokens:
- Markdown links `[text](/path.md)` become just `text`.
- The `## Links` / `## Related topics` navigation is dropped. It is useful for the graph, but it's noise for the LLM.

Concepts are added in retrieval order until `CONTEXT_CHARS` is used up.

In [13]:
def render(cid):
    c = concepts[cid]
    body = c["body"].split("\n## Links")[0].split("\n## Related topics")[0]
    body = LINK_RE.sub(r"\1", body)
    body = re.sub(r"\n{3,}", "\n\n", body).strip()[:MAX_CHARS_PER_CONCEPT]
    if c["type"] == "Paper Section":
        first, last = c["pages"] or [None, None]
        where = f"{c['paper']}, " + (f"p. {first}" if first == last else f"pp. {first}-{last}")
        header = f"### [{cid}] ({where})"
    else:
        header = f"### [{cid}] (curated {c['type']} page)"
    return header + "\n" + body


def build_context(picked, budget=CONTEXT_CHARS):
    parts, used, size = [], [], 0
    for cid, via, _ in picked:
        block = render(cid)
        if size + len(block) > budget:
            continue                      # skip what doesn't fit; a smaller one later may still fit
        parts.append(block)
        used.append(cid)
        size += len(block)
    return "\n\n".join(parts), used


context, used = build_context(retrieve("Which model was most accurate on TruthfulQA in HELM?"))
print(f"{len(used)} concepts, {len(context)} chars (~{len(context) // 4} tokens)\n")
print(context[:1500])

5 concepts, 17369 chars (~4342 tokens)

### [topics/instructgpt] (curated Model page)
# InstructGPT

## Summary

InstructGPT (Ouyang et al., 2022) improved GPT-3 for human alignment and "formally established a three-stage reinforcement learning from human feedback (RLHF) algorithm" (LLM survey 2.2, part 4, p. 10). The three stages are supervised fine-tuning, reward-model training and PPO. It is the direct ancestor of ChatGPT. See RLHF.

## What the papers say

### A Survey of Large Language Models
- Starts from the 175B GPT-3, with a 6B GPT-3 as reward model trained on labelers' rankings of outputs. Labelers wrote prompts and demonstrations for open QA, brainstorming, chatting and rewriting (5.2.3 RLHF, p. 38).
- OpenAI's papers rarely say "instruction tuning". They describe it as supervised fine-tuning on human demonstrations (2.2, part 4).
- Listed as a 175B model based on GPT-3, adapted with both instruction tuning and RLHF, released March 2022 (Table 1, 2.2, part 3).

### HELM
- HE

## 8. Ask the local LLM

The system prompt is strict on purpose. A 4B model will happily fill gaps from its own memory, so we tell it to use **only** the context and to cite concept IDs in square brackets. Then we can check every claim against the bundle.

`temperature: 0` makes answers repeatable. Ollama also returns token counts and timings, which we keep for comparing with GraphRAG later.

In [14]:
SYSTEM_PROMPT = '''You answer questions about five AI research papers using ONLY the knowledge below.
Each block starts with ### [concept-id]. Rules:
- Use only facts stated in the knowledge blocks. Do not add outside knowledge.
- After each claim, cite the concept id(s) you used in square brackets, e.g. [topics/calibration] or [papers/helm/4-4-calibration-and-uncertainty].
- Keep numbers, model names and percentages exactly as written.
- If the knowledge does not contain the answer, say: "The bundle does not contain this." and stop.
- Be concise: a short paragraph or a few bullets.'''


def ask_llm(question, context):
    start = time.perf_counter()
    response = requests.post(
        f"{OLLAMA_URL}/api/chat",
        json={
            "model": MODEL_NAME,
            "stream": False,
            "options": {"temperature": 0, "num_ctx": NUM_CTX, "num_predict": MAX_ANSWER_TOKENS},
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"Knowledge:\n\n{context}\n\n---\n\nQuestion: {question}"},
            ],
        },
        timeout=600,
    )
    response.raise_for_status()
    data = response.json()
    answer = re.sub(r"<think>.*?</think>", "", data["message"]["content"], flags=re.S).strip()
    stats = {
        "prompt_tokens": data.get("prompt_eval_count"),
        "answer_tokens": data.get("eval_count"),
        "generation_ms": round((time.perf_counter() - start) * 1000, 1),     # wall-clock time of the LLM call
    }
    return answer, stats

In [15]:
# warm-up: the first call loads the model into GPU memory (slow); later calls are much faster
start = time.perf_counter()
_ = ask_llm("Say OK.", "### [warmup]\nOK")
print(f"warm-up took {time.perf_counter() - start:.1f}s")

warm-up took 8.4s


## 9. Put it together: `ask()`

This runs the full pipeline and prints:
- the **retrieved** concepts, and why each was picked (`topic` / `section` / `link`);
- the **answer**;
- the **sources** the answer actually cited, with paper and pages, so you can open the PDF and check;
- **timing**: retrieval and generation in ms, and tokens.

In [16]:
def sources_for(answer, used):
    cited = [cid for cid in used if f"[{cid}]" in answer or cid in answer]
    lines = []
    for cid in cited:
        c = concepts[cid]
        if c["type"] == "Paper Section":
            first, last = c["pages"] or [None, None]
            pages = f"p. {first}" if first == last else f"pp. {first}-{last}"
            lines.append(f"{cid}  ->  {c['paper']}, {c['title']}, {pages}")
        else:
            lines.append(f"{cid}  ->  curated {c['type']} page: {c['title']}")
    return lines


def ask(question, show_context=False, **retrieve_kwargs):
    t0 = time.perf_counter()
    picked = retrieve(question, **retrieve_kwargs)
    context, used = build_context(picked)
    retrieval_ms = (time.perf_counter() - t0) * 1000

    answer, stats = ask_llm(question, context)

    print(f"Q: {question}\n")
    print("retrieved:")
    for cid, via, score in picked:
        print(f"   {via:8} {score:6.2f}  {cid}" + ("" if cid in used else "   (did not fit)"))
    if show_context:
        print("\n--- context sent to the model ---\n" + context + "\n--- end context ---")
    print("\nanswer:\n" + answer)
    print("\nsources cited:")
    for line in sources_for(answer, used) or ["(none - check the answer!)"]:
        print("   " + line)
    print(f"\ntiming: retrieval {retrieval_ms:.0f} ms | generation {stats['generation_ms']:.0f} ms | "
          f"tokens in/out {stats['prompt_tokens']}/{stats['answer_tokens']} | context {len(context)} chars")
    return {"question": question, "answer": answer, "retrieved": picked, "used": used,
            "retrieval_context": [render(cid) for cid in used],      # the exact blocks the model saw (for DeepEval)
            "retrieval_ms": round(retrieval_ms, 1), **stats}

## 10. Try it

The first four questions are the test queries from your `01_Chunking.ipynb`, so you can compare the answers with your Qdrant RAG.

In [17]:
r1 = ask("How does HELM measure calibration of language models?")

Q: How does HELM measure calibration of language models?

retrieved:
   topic     12.15  topics/mmlu
   topic     10.46  topics/calibration
   section   13.69  papers/helm/4-4-calibration-and-uncertainty
   section   12.65  papers/helm/8-1-meta-analysis-part-1
   link      10.57  papers/helm/c-2-calibration-and-uncertainty-part-1
   link       8.70  papers/helm/1-2-empirical-findings-part-1   (did not fit)

answer:
HELM measures calibration of language models using two primary metrics:  

1. **Expected Calibration Error (ECE)**: It groups model predictions into bins (10 bins in practice) based on confidence (e.g., 0.0–0.1, 0.1–0.2, ..., 0.9–1.0), computes the average gap between the model’s predicted confidence and its actual accuracy per bin, and sums these gaps weighted by bin size. This error quantifies how well the model’s stated confidence matches its real correctness. [papers/helm/4-4-calibration-and-uncertainty]  

2. **Selective Classification Accuracy**: It evaluates the accur

In [18]:
r2 = ask("What did the GPT-4 technical report say about predictable scaling?")

Q: What did the GPT-4 technical report say about predictable scaling?

retrieved:
   topic     22.13  topics/predictable-scaling
   topic     20.37  topics/scaling-laws
   section   21.64  papers/gpt-4-technical-report/3-predictable-scaling
   section   19.84  papers/gpt-4-technical-report/3-1-loss-prediction
   link      17.06  papers/gpt-4-technical-report/3-2-scaling-of-capabilities-on-humaneval
   link      11.90  papers/llm-survey/2-1-background-for-llms-part-2

answer:
The GPT-4 Technical Report stated that a large focus of the GPT-4 project was building a deep learning stack that scales predictably. This is because, for very large training runs like GPT-4, it is not feasible to do extensive model-specific tuning. To address this, they developed infrastructure and optimization methods that have very predictable behavior across multiple scales. These improvements allowed them to reliably predict some aspects of GPT-4's performance from smaller models trained using 1,000×–10,000× l

In [ ]:
r3 = ask("What is homogenization in foundation models and why is it risky?")

In [ ]:
r4 = ask("How did the Sparks of AGI paper test GPT-4 on theory of mind?")

Two questions that need **several papers at once**. This is where the curated topic pages should help, because they already combine what each paper says:

In [ ]:
r5 = ask("Which papers discuss hallucination, and what does each one say about it?")

In [ ]:
r6 = ask("What are the three steps of RLHF, and how did RLHF affect GPT-4's calibration?")

And one the bundle **can't** answer. A good RAG system should say so instead of inventing an answer:

In [ ]:
r7 = ask("How many parameters does Gemini Ultra have?")

## 11. Your turn

In [ ]:
question = "Which model was most accurate on TruthfulQA in HELM?"
result = ask(question, show_context=False)

In [ ]:
# summary of the seven runs
summary = pd.DataFrame([{k: r[k] for k in ("question", "retrieval_ms", "generation_ms", "prompt_tokens", "answer_tokens")}
                        for r in (r1, r2, r3, r4, r5, r6, r7)])
summary["question"] = summary["question"].str[:60]
display(summary)

## Save results for the comparison

This runs every question in `data/eval/eval_questions.jsonl` through `ask()` (quietly) and saves one line per question to `artifacts/results/okf.jsonl`. Each line holds:
- the **answer**
- **`retrieval_context`**: the exact blocks the model saw, in order
- **timing**: `retrieval_ms`, `generation_ms`
- **tokens**: `prompt_tokens`, `answer_tokens`

`08_evaluate_methods.ipynb` grades all four methods from these files with DeepEval, using one judge, so every method is judged the same way. Re-run this cell whenever you change the method or the questions.

In [ ]:
PROJECT_ROOT = BUNDLE_DIR.parent          # the folder that holds OKF_Bundle/

import contextlib
import io

METHOD = "okf"
EVAL_PATH = PROJECT_ROOT / "data" / "eval" / "eval_questions.jsonl"
RESULTS_PATH = PROJECT_ROOT / "artifacts" / "results" / f"{METHOD}.jsonl"
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)

eval_questions = [json.loads(line) for line in EVAL_PATH.open(encoding="utf-8") if line.strip()]
print(f"{len(eval_questions)} questions -> {RESULTS_PATH}")


def ask_quietly(question, attempts=2):
    # one retry, in case Ollama or Qdrant hiccups
    for attempt in range(attempts):
        try:
            with contextlib.redirect_stdout(io.StringIO()):          # ask() prints a lot; keep this cell readable
                return ask(question)
        except Exception as exc:
            if attempt == attempts - 1:
                raise
            print(f"   failed ({type(exc).__name__}: {str(exc)[:80]}), retrying in 10 s")
            time.sleep(10)


# written line by line: if it stops halfway, the answers so far are kept (08 shows the rest as missing)
with RESULTS_PATH.open("w", encoding="utf-8") as f:
    for n, item in enumerate(eval_questions, start=1):
        r = ask_quietly(item["question"])
        row = {"id": item["id"], "method": METHOD, "question": item["question"], "answer": r["answer"],
               "retrieval_context": r["retrieval_context"], "retrieval_ms": r["retrieval_ms"],
               "generation_ms": r["generation_ms"], "prompt_tokens": r["prompt_tokens"],
               "answer_tokens": r["answer_tokens"]}
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
        f.flush()
        print(f"[{n:2}/{len(eval_questions)}] {item['id']:5} retrieval {r['retrieval_ms']:6.0f} ms | "
              f"generation {r['generation_ms']:6.0f} ms | {r['answer'][:60]!r}")
print(f"saved {len(eval_questions)} results to {RESULTS_PATH.name}")

## 12. Experiments worth trying

1. **Turn off the graph step:** `ask(q, n_linked=0)`. Does the answer lose detail or page citations?
2. **Sections only (no wiki):** `ask(q, n_topics=0, n_sections=4)`. This is close to plain chunk RAG over sections. Compare it with the default.
3. **Wiki only:** `ask(q, n_sections=0, n_linked=0)`. Are the curated pages alone enough?
4. **See what the model saw:** `ask(q, show_context=True)`. When an answer is wrong, check whether the fact was in the context (a retrieval problem) or was there but misread (a model problem).
5. **Bigger context:** set `NUM_CTX = 12288` and `CONTEXT_CHARS = 30000` in cell 1 and re-run. Watch GPU memory in `nvidia-smi`.

### Things to know about this bundle
- Topic pages have `status: draft` and no `verified` field, so they are in OKF's **unverified** trust tier. They were written by an LLM (see `generated.by`) and every claim links to a section, so you can check them. When you've checked one, you can add `verified: {by: "human:jazz", at: "..."}` to its frontmatter.
- Section pages were split mechanically (`generated.by: process:okf-section-split`), so they contain the paper text as cleaned by your Chunking notebook. Tables and figure text can be messy.
- `index.md` (root and in each folder) is a table of contents you can open in VS Code's markdown preview to browse the bundle like a wiki. `log.md` records how the bundle was built.